# Практическая работа: Архитектуры агентов на LangGraph

В этой работе мы создадим 4 типа агентов с разной архитектурой и визуализируем каждый из них.

## Теория: как устроен LangGraph-агент

LangGraph строит агентов как **направленные графы** (StateGraph). Три ключевых понятия:

### 1. State (состояние)
Типизированный словарь, который передаётся между узлами. Обычно это `TypedDict` или `dataclass`.

```python
class State(TypedDict):
    messages: Annotated[list, add_messages]
```

`Annotated[list, add_messages]` — означает, что новые сообщения **добавляются** к списку, а не перезаписывают его.

### 2. Nodes (узлы)
Обычные Python-функции, которые принимают `state` и возвращают обновление состояния:

```python
def my_node(state: State) -> dict:
    result = llm.invoke(state["messages"])
    return {"messages": [result]}
```

### 3. Edges (рёбра)
Связи между узлами. Бывают:
- **Обычные** — `graph.add_edge("A", "B")` — всегда идём из A в B
- **Условные** — `graph.add_conditional_edges("A", router_fn, {...})` — выбор следующего узла по условию

### Сборка графа

```python
graph = StateGraph(State)
graph.add_node("agent", agent_fn)
graph.add_node("tool", tool_fn)
graph.set_entry_point("agent")
graph.add_conditional_edges("agent", should_continue, {"tool": "tool", "end": END})
graph.add_edge("tool", "agent")
app = graph.compile()
```

Готово — `app.invoke({"messages": [...]})` запускает граф.

### Визуализация
```python
from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))
```

---
## Установка и настройка

In [ ]:
! python.exe -m pip install --upgrade pip
! pip install langgraph langchain langchain-openai langchain-community duckduckgo-search numexpr

In [ ]:
from typing import  TypedDict
from IPython.display import Image, display

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool
from langchain.agents import create_agent


# ======== ВСТАВЬТЕ СВОИ ДАННЫЕ ========
API_KEY = "YOUR_API_KEY"
BASE_URL = 'YOUR_BASE_URL'
MODEL_NAME = "YOUR_MODEL_NAME"
# ======================================

llm = ChatOpenAI(api_key=API_KEY, base_url=BASE_URL, model=MODEL_NAME)

In [ ]:
def show_graph(app, title=""):
    if title:
        print(f"\n{'='*50}")
        print(f"  {title}")
        print(f"{'='*50}")
    display(Image(app.get_graph().draw_mermaid_png()))

---
# 1. Single Agent — LLM + Tools

Один узел: LLM оснащённая инструментами. Вызвали — получили ответ.

```
START → agent → END
```

In [ ]:
# --- Инструменты ---
@tool
def add(a: float, b: float) -> str:
    """Складывает два числа."""
    print(f"Called add({a}, {b})")
    return str(a + b)


@tool
def multiply(a: float, b: float) -> str:
    """Умножает два числа."""
    print(f"Called multiply({a}, {b})")
    return str(a * b)

# Добавьте вычитание и деление =)

tools_map = {
    "multiply": multiply,
    "add": add
}

# Создаем агента
llm_with_tools = llm.bind_tools([add, multiply])

In [ ]:
def tool_call(tool_answer, tools_map):
    return tools_map[tool_answer["name"]].invoke(tool_answer["args"])

In [ ]:
def single_agent(query: str, llm_with_tools, tools_map):
    response = llm_with_tools.invoke(query)
    
    if response.tool_calls:
        tools_result = ""

        for tool_response in response.tool_calls:
            tools_result += f"{tool_response["name"]}: {tool_call(tool_response, tools_map)}\n"
    
        answer_response = llm.invoke(f"Запрос: {query}. Результат вызова инструментов: {tools_result}. Дай ответ пользователю")
        return answer_response.content

    return response.content


In [ ]:
print(single_agent("Сколько будет 2 * 3? Вызывай тул", llm_with_tools, tools_map))

---
# 2. Workflow Agent — последовательная цепочка

Три шага строго друг за другом:
1. **Interpreter** — чистая LLM, упрощает и интерпретирует запрос пользователя
2. **Researcher** — LLM + search_tool, сама формулирует поисковый запрос и вызывает поиск
3. **Writer** — чистая LLM, генерирует финальный ответ

```
START → interpreter → researcher → writer → END
```

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

tools_map = {"duckduckgo_search": search_tool}

llm_with_tools = llm.bind_tools([search_tool])

In [ ]:
class WorkflowState(TypedDict):
    user_query: str
    interpreted_query: str
    search_results: str
    final_answer: str


# Шаг 1: LLM упрощает и интерпретирует запрос
def interpreter(state: WorkflowState):
    print("Called interpreter")
    response = llm.invoke(
        f"Упрости и интерпретируй следующий запрос пользователя. "
        f"Запрос: {state['user_query']}"
    )
    return {"interpreted_query": response.content}


# Шаг 2: Агент с search_tool — сам ищет информацию
def researcher(state: WorkflowState):
    print("Called researcher")

    result = single_agent(f"Найди информацию по теме: {state['interpreted_query']}", llm_with_tools, tools_map)

    return {"search_results": result}


# Шаг 3: LLM — генерирует финальный ответ
def writer(state: WorkflowState):
    print("Called writer")
    
    response = llm.invoke(
        f"На основе результатов поиска ответь на вопрос пользователя.\n\n"
        f"Вопрос: {state['user_query']}\n\n"
        f"Результаты поиска:\n{state['search_results']}"
    )

    return {"final_answer": response.content}


workflow = StateGraph(WorkflowState)
workflow.add_node("interpreter", interpreter)
workflow.add_node("researcher", researcher)
workflow.add_node("writer", writer)

workflow.set_entry_point("interpreter")
workflow.add_edge("interpreter", "researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", END)

workflow_agent = workflow.compile()
show_graph(workflow_agent, "Workflow Agent")

In [ ]:
response = workflow_agent.invoke({"user_query" : "Я забыл, как готовить борщ"})

In [ ]:
print(response["final_answer"])

---
# 3. Parallel Agent — параллельное выполнение

LLM-диспетчер получает запрос и сама решает, какой подзапрос отправить каждому из 3 агентов-исследователей. Каждый агент имеет свой `search_tool + LLM`. Агрегатор собирает результаты.

```
                  ┌→ researcher_1 (search + LLM) ─┐
START → dispatcher → researcher_2 (search + LLM) ─→ aggregator → END
                  └→ researcher_3 (search + LLM) ─┘
```

In [ ]:
from pydantic import BaseModel, Field

# 1. Определяем схему данных
class DispatcherTasks(BaseModel):
    task_1: str = Field(description="Первый уникальный поисковый запрос")
    task_2: str = Field(description="Второй уникальный поисковый запрос")
    task_3: str = Field(description="Третий уникальный поисковый запрос")


# 2. Создаем структурированную LLM (делаем это вне функции для эффективности)
dispatcher_llm = llm.with_structured_output(DispatcherTasks)

In [ ]:
search_tool = DuckDuckGoSearchRun()

tools_map = {"duckduckgo_search": search_tool}

researcher = llm.bind_tools([search_tool])

In [ ]:
import json

class ParallelState(TypedDict):
    user_query: str
    task_1: str
    task_2: str
    task_3: str
    result_1: str
    result_2: str
    result_3: str
    final_answer: str


# Шаг 1: LLM-диспетчер сама решает, какие запросы отправить агентам
def dispatcher(state: ParallelState):
    print("Called dispatcher")
    
    prompt = (
        f"Тебе нужно исследовать вопрос пользователя с помощью 3 параллельных агентов. "
        f"Придумай для каждого свой уникальный поисковый запрос, чтобы максимально полно раскрыть тему.\n\n"
        f"Вопрос пользователя: {state['user_query']}"
    )
    
    # 3. Вызываем модель — она вернет объект DispatcherTasks, а не строку
    tasks = dispatcher_llm.invoke(prompt)
    
    # Возвращаем данные в State (обращаемся к атрибутам объекта через точку)
    return {
        "task_1": tasks.task_1, 
        "task_2": tasks.task_2, 
        "task_3": tasks.task_3
    }


# 3 параллельных исследователя: search_tool + LLM
def researcher_1(state: ParallelState):
    print("Called researcher_1")
    
    summary = single_agent(f"Произведи поиск по запросу: '{state['task_1']}'. После кратко обобщи результаты поиска", 
                           researcher, 
                           tools_map)

    return {"result_1": summary}


def researcher_2(state: ParallelState):
    print("Called researcher_2")

    summary = single_agent(f"Произведи поиск по запросу: '{state['task_2']}'. После кратко обобщи результаты поиска", 
                           researcher, 
                           tools_map)

    return {"result_2": summary}


def researcher_3(state: ParallelState):
    print("Called researcher_3")
    
    summary = single_agent(f"Произведи поиск по запросу: '{state['task_3']}'. После кратко обобщи результаты поиска", 
                           researcher, 
                           tools_map)

    return {"result_3": summary}


# Агрегатор
def aggregator(state: ParallelState):
    print("Called aggregator")
    response = llm.invoke(
        f"Объедини результаты трёх исследований в один связный ответ на русском языке.\n\n"
        f"Вопрос пользователя: {state['user_query']}\n\n"
        f"Исследование 1:\n{state['result_1']}\n\n"
        f"Исследование 2:\n{state['result_2']}\n\n"
        f"Исследование 3:\n{state['result_3']}\n\n"
        f"Дай полный структурированный ответ."
    )
    return {"final_answer": response.content}


parallel = StateGraph(ParallelState)
parallel.add_node("dispatcher", dispatcher)
parallel.add_node("researcher_1", researcher_1)
parallel.add_node("researcher_2", researcher_2)
parallel.add_node("researcher_3", researcher_3)
parallel.add_node("aggregator", aggregator)

parallel.set_entry_point("dispatcher")
parallel.add_edge("dispatcher", "researcher_1")
parallel.add_edge("dispatcher", "researcher_2")
parallel.add_edge("dispatcher", "researcher_3")
parallel.add_edge("researcher_1", "aggregator")
parallel.add_edge("researcher_2", "aggregator")
parallel.add_edge("researcher_3", "aggregator")
parallel.add_edge("aggregator", END)

parallel_agent = parallel.compile()
show_graph(parallel_agent, "Parallel Agent")

In [ ]:
# Тест Parallel Agent
result = parallel_agent.invoke(
    {"user_query": "Сравни Python, Rust и Go для backend-разработки в 2025 году"}
)
print(result["final_answer"])

# 4. React Agent — рассуждение и действие
Агент выполняет цикл «Рассуждение → Действие» (Reason + Act): LLM генерирует мысль и выбирает инструмент, выполняет действие, наблюдает результат и повторяет до достижения цели.

```
START → react_agent (LLM + tools) ─┐
         ↑_________________________|
         │
         └→ END (при достижении цели)
```

In [ ]:
from langchain.agents import create_agent
import numexpr

search_tool = DuckDuckGoSearchRun()

@tool
def calculator(expression: str) -> str:
    """Вычисляет математическое выражение. Пример: '2 + 2 * 3'"""
    print("Called calculator")
    try:
        result = numexpr.evaluate(expression).item()
        return str(result)
    except Exception as e:
        return f"Ошибка: {e}"


react_agent = create_agent(
    model=llm,
    tools=[search_tool, calculator],
    system_prompt="Ты — полезный ассистент-исследователь. Используй поиск, чтобы находить актуальную информацию, "
           "и калькулятор для вычислений. Рассуждай пошагово. Отвечай на русском языке.",
)

show_graph(react_agent, "ReAct Agent")

In [ ]:
# Тест ReAct Agent
result = react_agent.invoke(
    {"messages": [HumanMessage(content="Какое население Москвы и Петербурга, и какова их сумма?")]}
)

print(result['messages'][-1].content)

# Практическое задание 

Необходимо создать агента, которые будет производить поиск товаров в базе данных (описана ниже), и давать ответ пользователю о наличии и описании товаров. 

Сделайте его в формате:
- Single agent
- Workflow agent
- Parallel agent
- ReAct agent

In [ ]:
# база товаров
PRODUCTS = {
    "iPhone 16 Pro": "Флагман Apple с чипом A18 Pro, 8 ГБ RAM, 256 ГБ, камера 48 МП, титановый корпус.",
    "Samsung Galaxy S25 Ultra": "Топовый Android с S Pen, 12 ГБ RAM, 200 МП камера, 5000 мА·ч, Snapdragon 8 Elite.",
    "Xiaomi 15": "Флагман с камерой Leica, 12 ГБ RAM, 5400 мА·ч, Snapdragon 8 Elite, привлекательная цена.",
    "Google Pixel 9": "Лучшая камера на Android, чистый софт, 7 лет обновлений. (нет в наличии)",
    "MacBook Air M3": "Тонкий лёгкий ноутбук с Apple M3, 16 ГБ RAM, автономность до 18 ч, без вентилятора.",
    "ASUS ROG Strix G16": "Игровой ноутбук: Intel i7, RTX 4060, 16 165 Гц, 1 ТБ SSD.",
    "Lenovo ThinkPad X1 Carbon Gen 12": "Бизнес-ультрабук с OLED 2.8K, 32 ГБ RAM, Intel Ultra 7.",
    "Sony WH-1000XM5": "Лучшее шумоподавление, премиальный звук, 30 ч работы, LDAC.",
    "AirPods Pro 2": "TWS с адаптивным шумоподавлением, 6 ч работы, интеграция с Apple.",
    "JBL Tune 520BT": "Бюджетные on-ear наушники, 57 ч работы, фирменный звук JBL.",
    "iPad Air M2": "Планшет с чипом M2, 11 Liquid Retina, поддержка Apple Pencil Pro.",
    "Samsung Galaxy Tab S9 FE": "Доступный планшет с S Pen в комплекте, защита IP68.",
}

In [ ]:
# здесь ваш код =)

In [ ]:
# ── Инструмент поиска товаров (переиспользуется во всех 4 паттернах) ──

@tool
def search_products(query: str) -> str:
    """Ищет товары в базе PRODUCTS по подстроке в названии или описании.
    Используй для поиска товаров по бренду, категории или характеристике.

    Args:
        query: поисковый запрос — часть названия, бренд или ключевое слово
               (например: "Apple", "наушники", "игровой")
    """
    query = query.lower()
    matches = [
        f"{name}: {description}"
        for name, description in PRODUCTS.items()
        if query in name.lower() or query in description.lower()
    ]
    if not matches:
        return f"По запросу '{query}' ничего не найдено"
    return "\n".join(matches)


tools_map = {"search_products": search_products}

In [ ]:
# Один цикл: LLM решает, вызывать ли инструмент, вызывает, получает результат,
# отвечает — без ветвления на этапы, всё внутри одной функции single_agent
llm_with_search = llm.bind_tools([search_products])

result = single_agent(
    "Есть ли в наличии наушники с шумоподавлением?",
    llm_with_search,
    tools_map,
)
print("=== Single Agent ===")
print(result)

In [ ]:
# Три жёстко прописанных шага: интерпретация запроса -> поиск -> финальный ответ.
# Маршрут ("что за чем идёт") задан кодом графа, а не решением модели

class ProductWorkflowState(TypedDict):
    user_query: str
    interpreted_query: str
    search_results: str
    final_answer: str


def interpreter(state: ProductWorkflowState):
    print("Called interpreter")
    response = llm.invoke(
        f"Упрости запрос пользователя до коротких ключевых слов для поиска товара. "
        f"Запрос: {state['user_query']}"
    )
    return {"interpreted_query": response.content}


def researcher(state: ProductWorkflowState):
    print("Called researcher")
    result = single_agent(
        f"Найди товары по запросу: {state['interpreted_query']}",
        llm_with_search,
        tools_map,
    )
    return {"search_results": result}


def writer(state: ProductWorkflowState):
    print("Called writer")
    response = llm.invoke(
        f"На основе найденных товаров ответь пользователю на его вопрос.\n\n"
        f"Вопрос: {state['user_query']}\n\n"
        f"Найденные товары:\n{state['search_results']}"
    )
    return {"final_answer": response.content}


product_workflow = StateGraph(ProductWorkflowState)
product_workflow.add_node("interpreter", interpreter)
product_workflow.add_node("researcher", researcher)
product_workflow.add_node("writer", writer)

product_workflow.set_entry_point("interpreter")
product_workflow.add_edge("interpreter", "researcher")
product_workflow.add_edge("researcher", "writer")
product_workflow.add_edge("writer", END)

product_workflow_agent = product_workflow.compile()
show_graph(product_workflow_agent, "Product Workflow Agent")

In [ ]:
response = product_workflow_agent.invoke(
    {"user_query": "Посоветуй игровой ноутбук"}
)
print(response["final_answer"])

In [ ]:
# Диспетчер сам придумывает 3 разных поисковых угла к одному вопросу
# (например: бренд, категория, конкретная характеристика),
# три researcher'а ищут параллельно (в графе — как параллельные рёбра из dispatcher),
# aggregator объединяет всё в единый ответ

class ProductDispatcherTasks(BaseModel):
    task_1: str = Field(description="Первый уникальный поисковый запрос по товарам")
    task_2: str = Field(description="Второй уникальный поисковый запрос по товарам")
    task_3: str = Field(description="Третий уникальный поисковый запрос по товарам")


product_dispatcher_llm = llm.with_structured_output(ProductDispatcherTasks)


class ProductParallelState(TypedDict):
    user_query: str
    task_1: str
    task_2: str
    task_3: str
    result_1: str
    result_2: str
    result_3: str
    final_answer: str


def product_dispatcher(state: ProductParallelState):
    print("Called dispatcher")
    prompt = (
        f"Придумай 3 разных поисковых запроса по базе товаров, чтобы полно ответить "
        f"на вопрос пользователя (например: под разные бренды, категории или характеристики).\n\n"
        f"Вопрос: {state['user_query']}"
    )
    tasks = product_dispatcher_llm.invoke(prompt)
    return {"task_1": tasks.task_1, "task_2": tasks.task_2, "task_3": tasks.task_3}


def product_researcher_1(state: ProductParallelState):
    print("Called researcher_1")
    summary = single_agent(
        f"Найди товары по запросу: '{state['task_1']}'", llm_with_search, tools_map
    )
    return {"result_1": summary}


def product_researcher_2(state: ProductParallelState):
    print("Called researcher_2")
    summary = single_agent(
        f"Найди товары по запросу: '{state['task_2']}'", llm_with_search, tools_map
    )
    return {"result_2": summary}


def product_researcher_3(state: ProductParallelState):
    print("Called researcher_3")
    summary = single_agent(
        f"Найди товары по запросу: '{state['task_3']}'", llm_with_search, tools_map
    )
    return {"result_3": summary}


def product_aggregator(state: ProductParallelState):
    print("Called aggregator")
    response = llm.invoke(
        f"Объедини результаты трёх поисков в один связный ответ на русском языке.\n\n"
        f"Вопрос: {state['user_query']}\n\n"
        f"Поиск 1:\n{state['result_1']}\n\n"
        f"Поиск 2:\n{state['result_2']}\n\n"
        f"Поиск 3:\n{state['result_3']}"
    )
    return {"final_answer": response.content}


product_parallel = StateGraph(ProductParallelState)
product_parallel.add_node("dispatcher", product_dispatcher)
product_parallel.add_node("researcher_1", product_researcher_1)
product_parallel.add_node("researcher_2", product_researcher_2)
product_parallel.add_node("researcher_3", product_researcher_3)
product_parallel.add_node("aggregator", product_aggregator)

product_parallel.set_entry_point("dispatcher")
product_parallel.add_edge("dispatcher", "researcher_1")
product_parallel.add_edge("dispatcher", "researcher_2")
product_parallel.add_edge("dispatcher", "researcher_3")
product_parallel.add_edge("researcher_1", "aggregator")
product_parallel.add_edge("researcher_2", "aggregator")
product_parallel.add_edge("researcher_3", "aggregator")
product_parallel.add_edge("aggregator", END)

product_parallel_agent = product_parallel.compile()
show_graph(product_parallel_agent, "Product Parallel Agent")

In [ ]:
result = product_parallel_agent.invoke(
    {"user_query": "Что есть из техники Apple и что из игровых устройств?"}
)
print(result["final_answer"])

In [ ]:
# Готовый ReAct-цикл из langchain.agents.create_agent:
# модель сама, итеративно, решает, сколько раз и как вызвать search_products,
# пока не сочтёт, что данных достаточно для ответа

product_react_agent = create_agent(
    model=llm,
    tools=[search_products],
    system_prompt=(
        "Ты — консультант магазина электроники. Используй search_products, чтобы "
        "находить товары в базе. Рассуждай пошагово, если нужно несколько поисков "
        "для полного ответа. Отвечай на русском языке."
    ),
)

show_graph(product_react_agent, "Product ReAct Agent")

In [ ]:
result = product_react_agent.invoke(
    {"messages": [HumanMessage(content="Сравни MacBook Air M3 и Lenovo ThinkPad, что лучше для работы?")]}
)
print(result["messages"][-1].content)